In [1]:
import os, sys, copy; from pathlib import Path; os.chdir(Path.home()/"IAA"); sys.path.insert(0, str(Path.home()/"IAA"))
import numpy as np, pandas as pd, itertools
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import src.pipeline as P, src.target as T
from src.config import PART2

cfg = copy.deepcopy(PART2); T.AMOUNTS_NET, T.ROW_COMPLETENESS = True, True
claims = T.load_claims(cfg["data_dir"])
pd.set_option("display.width", 200)

def cramers_v(a, b):
    t = pd.crosstab(a.astype(str), b.astype(str)).to_numpy(); n = t.sum()
    e = t.sum(1, keepdims=True) * t.sum(0, keepdims=True) / n
    chi2 = ((t - e) ** 2 / np.where(e > 0, e, 1)).sum()
    return np.sqrt(chi2 / n / max(min(t.shape) - 1, 1))

for line in cfg["lines"]:
    d = P.build_line(line, cfg, claims); tr = d["tr"]; f = d["feats"]
    num = [c for c in f if pd.api.types.is_numeric_dtype(tr[c]) and not pd.api.types.is_bool_dtype(tr[c]) and tr[c].nunique() > 2]
    cat = [c for c in f if c not in num]
    X = tr[num].astype(float); X = X.fillna(X.median()); X = X.loc[:, X.std() > 0]

    # 1. variable clusters: |Spearman r| >= 0.7
    R = X.corr(method="spearman").abs().fillna(0).to_numpy(); np.fill_diagonal(R, 1)
    lab = fcluster(linkage(squareform(1 - R, checks=False), "average"), t=0.3, criterion="distance")
    groups = [list(X.columns[lab == k]) for k in np.unique(lab) if (lab == k).sum() > 1]
    print(f"\n######## {line}: {len(num)} numeric, {len(cat)} categorical")
    print("variable clusters (|Spearman r| >= 0.7):", groups if groups else "none")

    # 2. Belsley condition indices and variance-decomposition proportions
    Z = np.column_stack([np.ones(len(X)), X.to_numpy()]); Z = Z / np.linalg.norm(Z, axis=0)
    _, s, Vt = np.linalg.svd(Z, full_matrices=False)
    ci = s.max() / s
    phi = (Vt.T ** 2) / s ** 2; vdp = phi / phi.sum(1, keepdims=True)          # rows = variables, cols = dimensions
    names = ["const"] + list(X.columns)
    for k in np.argsort(-ci):
        if ci[k] < 30: break
        load = [f"{names[j]} {vdp[j, k]:.2f}" for j in np.argsort(-vdp[:, k]) if vdp[j, k] >= 0.3]
        print(f"condition index {ci[k]:.0f}: variables sharing this dimension (VDP >= 0.3): {', '.join(load)}")

    # 3. categorical pairs: Cramér's V
    cv = sorted(((cramers_v(tr[a], tr[b]), a, b) for a, b in itertools.combinations(cat, 2)), reverse=True)[:6]
    print("top Cramér's V (categorical):", "; ".join(f"{a}~{b} {v:.2f}" for v, a, b in cv))

ValueError: underlying array is read-only

In [2]:
import os, sys, copy; from pathlib import Path; os.chdir(Path.home()/"IAA"); sys.path.insert(0, str(Path.home()/"IAA"))
import numpy as np, pandas as pd, itertools
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import src.pipeline as P, src.target as T
from src.config import PART2

cfg = copy.deepcopy(PART2); T.AMOUNTS_NET, T.ROW_COMPLETENESS = True, True
claims = T.load_claims(cfg["data_dir"])
pd.set_option("display.width", 200)

def cramers_v(a, b):
    t = pd.crosstab(a.astype(str), b.astype(str)).to_numpy(); n = t.sum()
    e = t.sum(1, keepdims=True) * t.sum(0, keepdims=True) / n
    chi2 = ((t - e) ** 2 / np.where(e > 0, e, 1)).sum()
    return np.sqrt(chi2 / n / max(min(t.shape) - 1, 1))

for line in cfg["lines"]:
    d = P.build_line(line, cfg, claims); tr = d["tr"]; f = d["feats"]
    num = [c for c in f if pd.api.types.is_numeric_dtype(tr[c]) and not pd.api.types.is_bool_dtype(tr[c]) and tr[c].nunique() > 2]
    cat = [c for c in f if c not in num]
    X = tr[num].astype(float); X = X.fillna(X.median()); X = X.loc[:, X.std() > 0]

    # 1. variable clusters: |Spearman r| >= 0.7
    R = X.corr(method="spearman").abs().fillna(0).to_numpy(copy=True); np.fill_diagonal(R, 1)
    lab = fcluster(linkage(squareform(np.clip(1 - R, 0, None), checks=False), "average"), t=0.3, criterion="distance")
    groups = [list(X.columns[lab == k]) for k in np.unique(lab) if (lab == k).sum() > 1]
    print(f"\n######## {line}: {len(num)} numeric, {len(cat)} categorical")
    print("variable clusters (|Spearman r| >= 0.7):", groups if groups else "none")

    # 2. Belsley condition indices and variance-decomposition proportions
    Z = np.column_stack([np.ones(len(X)), X.to_numpy(copy=True)]); Z = Z / np.linalg.norm(Z, axis=0)
    _, s, Vt = np.linalg.svd(Z, full_matrices=False)
    ci = s.max() / s
    phi = (Vt.T ** 2) / s ** 2; vdp = phi / phi.sum(1, keepdims=True)          # rows = variables, cols = dimensions
    names = ["const"] + list(X.columns)
    for k in np.argsort(-ci):
        if ci[k] < 30: break
        load = [f"{names[j]} {vdp[j, k]:.2f}" for j in np.argsort(-vdp[:, k]) if vdp[j, k] >= 0.3]
        print(f"condition index {ci[k]:.0f}: variables sharing this dimension (VDP >= 0.3): {', '.join(load)}")

    # 3. categorical pairs: Cramér's V
    cv = sorted(((cramers_v(tr[a], tr[b]), a, b) for a, b in itertools.combinations(cat, 2)), reverse=True)[:6]
    print("top Cramér's V (categorical):", "; ".join(f"{a}~{b} {v:.2f}" for v, a, b in cv))


######## motor: 21 numeric, 22 categorical
variable clusters (|Spearman r| >= 0.7): [['vehicle_value_group', 'engine_size_cc'], ['freezing_days', 'dry_summer_index', 'mean_temperature_c']]
condition index 1033: variables sharing this dimension (VDP >= 0.3): mean_temperature_c 1.00, const 0.99, freezing_days 0.98, dry_summer_index 0.97, wildfire_index 0.63
condition index 69: variables sharing this dimension (VDP >= 0.3): 
condition index 48: variables sharing this dimension (VDP >= 0.3): engine_size_cc 0.81, vehicle_value_group 0.74
condition index 36: variables sharing this dimension (VDP >= 0.3): rainfall_mm 0.81
top Cramér's V (categorical): driver_age_band~marital_status 0.28; driver_age_band~cover_type 0.18; driver_age_band~body_type 0.16; driver_age_band~new_business 0.12; named_young_driver~marital_status 0.12; driver_age_band~named_young_driver 0.12
12:40:37 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
12:40:37 buildings: fire priced separately at 9

In [3]:
import os; from pathlib import Path; os.chdir(Path.home()/"IAA")
txt = """### E27: collinear variable groups replaced by one variable (Oct 25, notebook 30) — rule fixed before running
- Diagnosis (no target used): VIF on numeric features (Oct 24, 02:25) mean_temperature_c 430-460, freezing_days 134-144, dry_summer_index
  88-93, wildfire_index 5.5; engine_size_cc ~ vehicle_value_group r 0.88; plus variable clustering and Belsley condition indices (notebook 30).
- Groups by reasoning: climate (mean_temperature_c, freezing_days, dry_summer_index, wildfire_index; all lines: address averages that
  measure how cold / dry a place is); vehicle size and value (engine_size_cc, vehicle_value_group; motor). Contents prem_rel ~ sum_insured
  (r 0.90) left out (E23 adopted yesterday; the insurer prices partly from the sum insured).
- Candidates: A = keep the group's most central variable (highest mean |r| with the others), drop the rest; B = replace the group by its
  first principal component (standardised, fitted on emerging training rows, same transform on the developed table and the test).
- Rule (Part 1 simplification rule), per line on the 2024 fold: adopt if D2 falls by no more than 0.0005 and neither tail decile A/E moves
  more than 0.1 further from 1. If A and B both pass, A (simpler). The model stays v7 unless a line passes.
"""
p = Path("DECISIONS.md"); d = p.read_text(encoding="utf-8")
a = "## Assumptions register"; assert d.count(a) == 1 and "### E27:" not in d
p.write_text(d.replace(a, txt + "\n" + a), encoding="utf-8"); print("recorded")

recorded


In [4]:
import os, sys, copy; from pathlib import Path; os.chdir(Path.home()/"IAA"); sys.path.insert(0, str(Path.home()/"IAA"))
import numpy as np, pandas as pd
import src.pipeline as P, src.target as T
from src.config import PART2
from src.evaluate import evaluate

GROUPS = {"climate_index": ["mean_temperature_c", "freezing_days", "dry_summer_index", "wildfire_index"],
          "vehicle_size_index": ["engine_size_cc", "vehicle_value_group"]}
BASE_D2 = {"motor": 0.0373, "buildings": 0.0684, "contents": 0.0553}

cfg = copy.deepcopy(PART2); T.AMOUNTS_NET, T.ROW_COMPLETENESS = True, True
claims = T.load_claims(cfg["data_dir"])

def fit_groups(df, feats):
    spec = {}
    for g, cols in GROUPS.items():
        cols = [c for c in cols if c in feats and c in df]
        if len(cols) < 2: continue
        X = df[cols].astype(float); med = X.median(); X = X.fillna(med); mu, sd = X.mean(), X.std()
        C = np.corrcoef(((X - mu) / sd).T.values); w, V = np.linalg.eigh(C); v = V[:, -1]
        v = v * np.sign(v[np.argmax(np.abs(v))])
        rep = cols[int(np.argmax(np.abs(C).sum(0) - 1))]
        spec[g] = dict(cols=cols, med=med, mu=mu, sd=sd, v=v, share=w[-1] / w.sum(), rep=rep)
    return spec

def transform(df, spec, method):
    df = df.copy(); drop, add = [], []
    for g, s in spec.items():
        if method == "B":
            X = df[s["cols"]].astype(float).fillna(s["med"])
            df[g] = ((X - s["mu"]) / s["sd"]).to_numpy() @ s["v"]; drop += s["cols"]; add.append(g)
        else:
            drop += [c for c in s["cols"] if c != s["rep"]]
    return df, drop, add

def metrics(line, d, va, rate):
    w = va.weight.to_numpy(); r = np.asarray(rate, float)
    act_all = (va.target_all if "target_all" in va else va.target).to_numpy()
    act_mod = (va.target_modelled if "target_modelled" in va else va.target).to_numpy()   # rule b for buildings
    d2 = evaluate(pd.Series(act_all), (r + d.get("sep_rate", 0.0)) * w, pd.Series(w), show=False)["D2"]
    lf = evaluate(pd.Series(act_mod), r * w, pd.Series(w), show=False)["lift"]["A/E"]
    return {"D2": d2, "low": lf.iloc[0], "top": lf.iloc[-1]}

rows = []
for line in cfg["lines"]:
    d = P.build_line(line, cfg, claims); feats = d["feats"]
    tr, va = d["tr"][d["tr"].year == 2023], d["tr"][d["tr"].year == 2024]
    dev_mode = line in cfg.get("developed_training", {})
    dev0 = P.developed_table(line, cfg).copy() if dev_mode else None
    spec = fit_groups(tr, feats)
    for g, s in spec.items():
        print(f"{line} {g}: {s['cols']} | first component explains {s['share']:.0%} | central variable: {s['rep']}")
    for method in ["current", "A", "B"]:
        if method == "current":
            tr2, va2, f2 = tr, va, feats
            if dev_mode: P._DEV[line] = dev0
        else:
            tr2, drop, add = transform(tr, spec, method); va2, _, _ = transform(va, spec, method)
            f2 = [c for c in feats if c not in drop] + add
            if dev_mode: P._DEV[line] = transform(dev0, spec, method)[0]
        rate, _ = P.predict_rate(line, cfg, tr2, va2, f2)
        rows.append({"line": line, "model": method, "features": len(f2), **metrics(line, d, va2, rate)})
    if dev_mode: P._DEV[line] = dev0
    P.log(f"{line} done")

t = pd.DataFrame(rows); print("\n" + t.round(4).to_string(index=False))
print("\nRule: D2 falls by no more than 0.0005, tails no more than 0.1 further from 1; A preferred if both pass")
for line in cfg["lines"]:
    b = t[(t.line == line) & (t.model == "current")].iloc[0]
    assert abs(b.D2 - BASE_D2[line]) < 0.0005, f"{line}: current D2 differs from v7 fold"
    ok = {}
    for m in ["A", "B"]:
        n = t[(t.line == line) & (t.model == m)].iloc[0]
        ok[m] = (n.D2 >= b.D2 - 0.0005) and all(abs(n[k] - 1) <= abs(b[k] - 1) + 0.1 for k in ["low", "top"])
        print(f"  {line} {m}: D2 {n.D2 - b.D2:+.4f}, features {int(b.features)} -> {int(n.features)} -> {'passes' if ok[m] else 'fails'}")
    print(f"  {line}: {'adopt A' if ok['A'] else 'adopt B' if ok['B'] else 'keep current'}")

12:41:04 motor: developed training table 979,968 rows
motor climate_index: ['mean_temperature_c', 'freezing_days', 'dry_summer_index', 'wildfire_index'] | first component explains 79% | central variable: mean_temperature_c
motor vehicle_size_index: ['engine_size_cc', 'vehicle_value_group'] | first component explains 94% | central variable: engine_size_cc
12:42:08 motor done
12:42:08 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
12:42:08 buildings: fire priced separately at 97.05 per exposure (developed)
12:42:09 buildings: own GLM for ['flood']; main GLM on the other perils
buildings climate_index: ['mean_temperature_c', 'freezing_days', 'dry_summer_index', 'wildfire_index'] | first component explains 79% | central variable: mean_temperature_c
12:42:14 buildings: developed training table 707,726 rows
12:42:37 buildings done
12:42:37 contents: sum_insured x1000 entries divided by 1000: train 57, test 135
12:42:42 contents: developed training table 721,617 rows

In [5]:
import os; from pathlib import Path; os.chdir(Path.home()/"IAA")
txt = """- **E27 result (Oct 25, 12:43):** first component of the climate group explains 79% (central variable mean_temperature_c);
  vehicle group 94% (central engine_size_cc).
  | line | current | A (keep central) | B (first component) | verdict |
  | motor | 0.0373 (tails 0.80 / 0.89) | 0.0371 (0.79 / 0.88) | 0.0382 (0.74 / 0.94) | both pass -> A |
  | buildings | 0.0681 (0.58 / 1.02) | 0.0694 (0.57 / 1.01) | 0.0702 (0.53 / 1.02) | both pass -> A |
  | contents | 0.0550 (0.85 / 0.98) | 0.0558 (0.64 / 0.96) | 0.0563 (0.62 / 0.98) | both fail (lowest decile -0.21) -> keep |
  Contents: the climate variables separate the safest customers there; removing them moves the lowest decile 0.85 -> 0.64.
- **Decision (Woojin, Oct 25):** adopt A in motor and buildings as a simplification (7 fewer variables) under the rule fixed
  before the run. This overrides the Oct 24 freeze for this change only, with a guard. Dropped: motor freezing_days,
  dry_summer_index, wildfire_index, vehicle_value_group; buildings freezing_days, dry_summer_index, wildfire_index.
- **p2_v9 expected public, written before submitting: 0.1426-0.1442** (validation change about +0.0003 pooled; within public
  noise). **Guard: if public < 0.1420, v7 stays the final.** Pipeline validation must reproduce motor 0.0371 and buildings 0.0694.
"""
p = Path("DECISIONS.md"); d = p.read_text(encoding="utf-8")
a = "## Assumptions register"; assert d.count(a) == 1 and "### E27:" in d and "E27 result" not in d
p.write_text(d.replace(a, txt + "\n" + a), encoding="utf-8"); print("recorded")

recorded


In [6]:
import os; from pathlib import Path; os.chdir(Path.home()/"IAA")
p = Path("src/config.py"); s = p.read_text(encoding="utf-8")
anchor = 'PART2["premium_lines"] = ["contents"]'
add = '''
# Oct 25 (E27): collinear climate / vehicle groups reduced to their central variable (motor, buildings; contents fails tails)
PART2["drop"]["motor"] = PART2["drop"]["motor"] + ["freezing_days", "dry_summer_index", "wildfire_index", "vehicle_value_group"]
PART2["drop"]["buildings"] = PART2["drop"]["buildings"] + ["freezing_days", "dry_summer_index", "wildfire_index"]
'''
assert s.count(anchor) == 1 and "E27" not in s
i = s.index(anchor); j = s.index("\n", i) + 1
p.write_text(s[:j] + add + s[j:], encoding="utf-8"); print("patched")

patched


In [7]:
import os, sys, importlib; from pathlib import Path; os.chdir(Path.home()/"IAA"); sys.path.insert(0, str(Path.home()/"IAA"))
import src.config as C; importlib.reload(C)
import src.pipeline as P; importlib.reload(P)
from src.config import PART2
assert "wildfire_index" in PART2["drop"]["buildings"] and "wildfire_index" not in C.PART1["drop"]["buildings"]

res = P.validate(PART2)
print(res)

12:48:52 motor: developed training table 979,968 rows
12:49:12 validate motor 2024: D2 0.0371  A/E 1.074  new 0.996  renewal 1.084
12:49:12 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
12:49:12 buildings: fire priced separately at 97.05 per exposure (developed)
12:49:13 buildings: own GLM for ['flood']; main GLM on the other perils
12:49:17 buildings: developed training table 707,726 rows
12:49:25 validate buildings 2024: D2 0.0694  A/E 0.907  new 0.683  renewal 0.930
12:49:25 contents: sum_insured x1000 entries divided by 1000: train 57, test 135
12:49:28 contents: developed training table 721,617 rows
12:49:50 validate contents 2024: D2 0.0550  A/E 0.930  new 1.340  renewal 0.889
12:49:50 validate pooled 2024: D2 0.1084  A/E 1.005
           line  year        D2       A/E       low       top   A/E new  A/E renewal     low b     top b
0         motor  2024  0.037138  1.074448  0.787420  0.878193  0.995518     1.084409       NaN       NaN
1     buildings  20

In [8]:
out = "submissions/p2_v9_e27_simplified.csv"
P.run(PART2, out)
P.compare("submissions/p2_v7_leverage.csv", out)

12:51:09 motor: GLM on developed data (38 features), emerging level x0.8022; Pareto, alpha 2.05 borrowed: 3 claims above 100,000, loading 1.52 -> factor 1.0031; broker-book load x1.3 on 76% of rows; mean rate 588.80; total 58,095,494; completeness {2023: 0.982, 2024: 0.805} (22s)
12:51:09 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
12:51:10 buildings: fire priced separately at 97.05 per exposure (developed)
12:51:10 buildings: own GLM for ['flood']; main GLM on the other perils
12:51:19 buildings: GLM, k=1.0542 + flood GLM on combined data (17 features, alpha 0.5, level x0.7931); no loading; + separate peril rate 97.05; broker-book load x1.3 on 76% of rows; mean rate 362.94; total 21,184,432; completeness {2023: 1.0, 2024: 0.95} (9s)
12:51:19 contents: sum_insured x1000 entries divided by 1000: train 57, test 135
12:51:43 contents: GLM on combined data (46 features), emerging level x0.9654; no loading; broker-book load x1.3 on 76% of rows; mean rate 100.82;

,row_id,prediction_a,prediction_b
0,M00109296,352.066291,363.591182
1,M00109297,303.349896,305.796343
2,M00109298,283.922644,279.358379
3,M00109299,678.639567,650.398914
4,M00109300,334.268552,338.056809
...,...,...,...
209058,C00318354,24.901636,24.901636
209059,C00318355,59.078670,59.078670
209060,C00318356,37.999701,37.999701
209061,C00318357,87.774112,87.774112


In [9]:
import pandas as pd, numpy as np
a = pd.read_csv("submissions/p2_v7_leverage.csv"); b = pd.read_csv("submissions/p2_v9_e27_simplified.csv")
id_col, val = a.columns[:2]
m = a.merge(b, on=id_col, suffixes=("_v7", "_v9")); assert len(m) == len(a) == len(b)
m["line"] = m[id_col].str[0].map({"M": "motor", "B": "buildings", "C": "contents"})
r = m[f"{val}_v9"] / m[f"{val}_v7"] - 1
out = m.assign(r=r).groupby("line").apply(lambda g: pd.Series({
    "rows": len(g), "total_change_%": 100 * (g[f"{val}_v9"].sum() / g[f"{val}_v7"].sum() - 1),
    "corr": np.corrcoef(g[f"{val}_v7"], g[f"{val}_v9"])[0, 1],
    "row_change_p5_%": 100 * g.r.quantile(0.05), "row_change_p95_%": 100 * g.r.quantile(0.95)}))
print(out.round(4)); print("all lines total change %:", round(100 * (m[f"{val}_v9"].sum() / m[f"{val}_v7"].sum() - 1), 3))

              rows  total_change_%    corr  row_change_p5_%  row_change_p95_%
line                                                                         
buildings  52295.0          0.0658  0.9999          -2.4778            2.5430
contents   66247.0          0.0000  1.0000           0.0000            0.0000
motor      90521.0          0.0244  0.9974          -5.8714            7.4005
all lines total change %: 0.032


In [10]:
import os; from pathlib import Path; os.chdir(Path.home()/"IAA")
txt = """- Pipeline validation reproduced the experiment: motor 0.0371, buildings 0.0694 (rule-b tails 0.571 / 1.013), contents 0.0550,
  pooled 0.1084. v9 vs v7 totals: motor +0.02%, buildings +0.07%, contents 0 (identical), all +0.03%; correlation motor 0.9974,
  buildings 0.9999; row changes (5-95%) motor -5.9% to +7.4%, buildings -2.5% to +2.5%.
- **p2_v9 public (Oct 25, 12:56): 0.14477**, +0.0014 vs v7; above the range written before submitting (0.1426-0.1442), most likely
  public noise on top of a small real gain (validation +0.0003 pooled). Guard passed. **p2_v9_e27_simplified is the Part 2 final.**
  Reported as: seven collinear variables removed without loss of accuracy, not as a score improvement.
"""
p = Path("DECISIONS.md"); d = p.read_text(encoding="utf-8")
a = "## Assumptions register"; assert d.count(a) == 1 and "E27 result" in d and "p2_v9 public" not in d
p.write_text(d.replace(a, txt + "\n" + a), encoding="utf-8"); print("recorded")

recorded


In [12]:
import os; from pathlib import Path; os.chdir(Path.home()/"IAA")
rows = """| 115 | H | Team asked whether correlated variables could be found without the target, merged by reasoning into one, and re-run. | Rule fixed before running (E27): variable clusters (Spearman), Belsley condition indices, candidates A (keep the group's central variable) vs B (first principal component), Part 1 simplification rule on the 2024 fold; pipeline validation had to reproduce the experiment; public guard (< 0.1420 keeps v7). | Climate group first component 79%, vehicle group 94%. Motor and buildings: A adopted (7 variables removed); contents failed (lowest decile 0.85 -> 0.64). Pipeline reproduced 0.0371 / 0.0694 / 0.0550. v9 public 0.14477 (+0.0014; above the range written beforehand, validation gain only +0.0003, so mostly noise). The same idea had been rejected on Oct 23 under the stability rule (99); a different justification made it testable. AI's first diagnostic cell failed on a read-only array (pandas copy semantics); fixed. |
| 116 | H | Team asked for last year's winning method and a comparison with ours. | AI could not recall the methods and said so instead of answering from memory; found the winner's Kaggle notebook (IAA News Brief, Jan 2026) and read it in full. | Last year's task was house prices (RMSLE), not insurance. Methods: LightGBM / XGBoost / CatBoost voting ensemble (ours: E10, no gain), separate simple model for a structural outlier segment (ours: fire and flood split), MID surrogate distilling the ensemble into a GA2M rating structure (not tried; report candidate). Random 5-fold CV there vs out-of-time validation here (year shifts make random folds leak). |
"""
p = Path("AI_USE_LOG.md"); d = p.read_text(encoding="utf-8")
lines = d.split("\n")
idx = [i for i, l in enumerate(lines) if l.startswith("| 114 |")]
assert len(idx) == 1 and "| 115 |" not in d
lines.insert(idx[0] + 1, rows.rstrip("\n"))
p.write_text("\n".join(lines), encoding="utf-8"); print("AI_USE_LOG 115-116 recorded")

AI_USE_LOG 115-116 recorded


In [13]:
import os, subprocess; from pathlib import Path; os.chdir(Path.home()/"IAA")
subprocess.run(["git", "checkout", "--", "README.md"], check=True)   # drop the stray local rows first
p = Path("README.md"); s = p.read_text(encoding="utf-8")
anchor = "## Tests"
add = """## Reproduce the Part 2 final submission (v9, public 0.14477)
```bash
python -m venv .venv && source .venv/bin/activate
pip install -r requirements-lock.txt          # exact environment used for the submissions
# competition data (not committed): Part 1 parquet files in data/, Part 2 in data_part2/emerging and data_part2/development
python -m src.pipeline --market part2 --out submissions/p2_v9_e27_simplified.csv
python -m src.pipeline --market part2 --validate   # out-of-time check: 2024 D2 motor 0.0371, buildings 0.0694, contents 0.0550
```
Settings live in `src/config.py` (`PART2`); every change and its evidence is in `DECISIONS.md`.
Checked on Oct 25 from a fresh clone: identical to the submitted file (max abs difference 0).

"""
assert s.count(anchor) == 1 and "Reproduce the Part 2 final" not in s
p.write_text(s.replace(anchor, add + anchor), encoding="utf-8"); print("README updated")

README updated
